# India MonsoonBench Quickstart

This lightweight notebook shows how to inspect the released India MonsoonBench archive. It is designed to run even before the large archive is extracted: metadata tables from the GitHub repository are loaded first, and patch-array loading is attempted only if a regional dataset folder is available locally.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

release_root = repo / "India_MonsoonBench_DataRelease"
patch_root = release_root / "patch_datasets"

print("Repository root:", repo)
print("Expected extracted archive:", release_root)

## 2. Regional Dataset Summary

This table is committed in the GitHub repository and can be inspected without downloading the full archive.

In [ ]:
summary_path = repo / "results" / "metadata" / "regional_dataset_summary.csv"
summary = pd.read_csv(summary_path)
summary

## 3. State-Level Patch Counts

In [ ]:
counts_path = repo / "results" / "metadata" / "regional_state_patch_counts.csv"
state_counts = pd.read_csv(counts_path)
state_counts.head(20)

## 4. Locate an Extracted Patch Dataset

After downloading and extracting the archive, the expected layout is:

```text
India_MonsoonBench_DataRelease/patch_datasets/<regional_dataset>/
```

If you are running this notebook inside the original working directory, it also checks for regional patch datasets at the repository root.

In [ ]:
candidate_dirs = [
    patch_root / "patch_dataset_monthly_ar_northwest_himalayan_full",
    patch_root / "patch_dataset_monthly_ar_central_monsoon_core_full_bihar",
    patch_root / "patch_dataset_monthly_ar_south_peninsular_deccan_full",
    patch_root / "patch_dataset_monthly_ar_east_northeast_humid_orographic_full_assam",
    repo / "patch_dataset_monthly_ar_northwest_himalayan_full",
    repo / "patch_dataset_monthly_ar_central_monsoon_core_full_bihar",
    repo / "patch_dataset_monthly_ar_south_peninsular_deccan_full",
    repo / "patch_dataset_monthly_ar_east_northeast_humid_orographic_full_assam",
]

dataset_dir = next((p for p in candidate_dirs if (p / "X.npy").exists()), None)
dataset_dir

## 5. Load Shapes, Config, and Splits

The arrays are opened with `mmap_mode="r"` so the full `X.npy` tensor is not loaded into memory.

In [ ]:
if dataset_dir is None:
    print("No extracted patch dataset found. Download and extract the archive, then rerun this cell.")
else:
    config = json.loads((dataset_dir / "config.json").read_text())
    samples = pd.read_csv(dataset_dir / "samples.csv")
    X = np.load(dataset_dir / "X.npy", mmap_mode="r")
    y = np.load(dataset_dir / "y.npy", mmap_mode="r")
    train_idx = np.load(dataset_dir / "train_idx.npy")
    val_idx = np.load(dataset_dir / "val_idx.npy")
    test_idx = np.load(dataset_dir / "test_idx.npy")

    print("Dataset:", dataset_dir.name)
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Splits:", len(train_idx), len(val_idx), len(test_idx))
    print("Config keys:", sorted(config.keys()))
    display(samples.head())

## 6. Class Distribution

In [ ]:
class_names = ["Scarcity", "Deficit", "Normal", "Excess", "Large Excess"]

if dataset_dir is not None:
    counts = pd.Series(np.asarray(y), name="class_id").value_counts().sort_index()
    counts.index = [class_names[int(i)] for i in counts.index]
    ax = counts.plot(kind="bar", figsize=(7, 3), color="#2b83ba")
    ax.set_title(f"Rainfall class distribution: {dataset_dir.name}")
    ax.set_ylabel("Samples")
    ax.set_xlabel("Rainfall anomaly class")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    display(counts.to_frame("num_samples"))

## 7. Visualize One Patch Channel

This cell displays one channel from the first test sample. Channel interpretation depends on the extraction config, but the benchmark uses 6 dynamic predictors across 3 input months plus static elevation, for 19 channels total.

In [ ]:
if dataset_dir is not None:
    sample_index = int(test_idx[0])
    channel = 0
    patch = np.asarray(X[sample_index, channel])

    plt.figure(figsize=(4, 4))
    plt.imshow(patch, cmap="viridis")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(f"Sample {sample_index}, channel {channel}, label {class_names[int(y[sample_index])]}")
    plt.axis("off")
    plt.show()

## 8. Next Steps

To train a small baseline from the repository root:

```bash
python train_patch_baselines.py \
  --dataset-dir India_MonsoonBench_DataRelease/patch_datasets/patch_dataset_monthly_ar_northwest_himalayan_full \
  --output-dir baseline_runs/tutorial_convlstm_northwest \
  --model convlstm \
  --loss ce \
  --epochs 5 \
  --batch-size 16 \
  --disable-early-stopping
```

For full benchmark reproduction, use the commands in `REPRODUCIBILITY.md`.